<a href="https://colab.research.google.com/github/shafiq73/2024/blob/main/Mini_YData_Profiling_Function2_(with_Alerts).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install plotly jinja2 numpy

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from jinja2 import Template
def smart_profiling_report(df, filename="Smart_EDA_Report.html"):


    # ========== BASIC INFO ==========
    shape = df.shape
    dtypes_html = df.dtypes.to_frame("Data Type").to_html()
    describe_html = df.describe(include='all').to_html()

    # ========== MISSING VALUES ==========
    missing = (df.isnull().sum() / len(df) * 100).reset_index()
    missing.columns = ['Column', 'Missing %']
    fig_missing = px.bar(missing, x='Column', y='Missing %',
                         title="Missing Values %")
    missing_plot = fig_missing.to_html(full_html=False)

    # ========== CORRELATION ==========
    corr = df.corr(numeric_only=True)
    fig_corr = px.imshow(corr, text_auto=True,
                         title="Correlation Heatmap")
    corr_plot = fig_corr.to_html(full_html=False)

    # Detect highly correlated pairs
    high_corr = []
    for i in corr.columns:
        for j in corr.columns:
            if i != j and abs(corr.loc[i, j]) > 0.8:
                high_corr.append(f"{i} & {j} : {corr.loc[i,j]:.2f}")

    # ========== HISTOGRAMS ==========
    histograms = ""
    for col in df.select_dtypes(include=np.number).columns:
        fig = px.histogram(df, x=col, title=f"Histogram of {col}")
        histograms += fig.to_html(full_html=False)

    # ========== CATEGORICAL BARS ==========
    bars = ""
    for col in df.select_dtypes(include='object').columns:
        vc = df[col].value_counts().head(10).reset_index()
        vc.columns = [col, 'Count']
        fig = px.bar(vc, x=col, y='Count',
                     title=f"Top Categories of {col}")
        bars += fig.to_html(full_html=False)

    # ========== ALERTS / WARNINGS ==========
    alerts = []

    # Constant columns
    const_cols = [col for col in df.columns if df[col].nunique() == 1]
    if const_cols:
        alerts.append(f"Constant Columns: {const_cols}")

    # High cardinality
    high_card = [col for col in df.columns if df[col].nunique() > 50]
    if high_card:
        alerts.append(f"High Cardinality Columns: {high_card}")

    # Missing columns
    miss_cols = missing[missing['Missing %'] > 0]['Column'].tolist()
    if miss_cols:
        alerts.append(f"Columns with Missing Values: {miss_cols}")

    # High correlation
    if high_corr:
        alerts.append(f"Highly Correlated Pairs: {high_corr}")

    alerts_html = "<br>".join(alerts) if alerts else "No major warnings detected."

    # ========== HTML TEMPLATE ==========
    html_template = """
    <html>
    <head>
        <title>Smart EDA Report</title>
    </head>
    <body>
        <h1>📊 Smart Auto EDA Report</h1>

        <h2>Dataset Shape</h2>
        <p>{{shape}}</p>

        <h2>⚠️ Alerts & Warnings</h2>
        <p>{{alerts}}</p>

        <h2>Data Types</h2>
        {{dtypes}}

        <h2>Statistical Summary</h2>
        {{describe}}

        <h2>Missing Values</h2>
        {{missing_plot}}

        <h2>Correlation Heatmap</h2>
        {{corr_plot}}

        <h2>Numeric Distributions</h2>
        {{histograms}}

        <h2>Categorical Distributions</h2>
        {{bars}}

    </body>
    </html>
    """

    template = Template(html_template)
    html = template.render(shape=shape,
                           alerts=alerts_html,
                           dtypes=dtypes_html,
                           describe=describe_html,
                           missing_plot=missing_plot,
                           corr_plot=corr_plot,
                           histograms=histograms,
                           bars=bars)

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Smart EDA report saved as {filename}")



In [ ]:
import kagglehub
path = kagglehub.dataset_download("lava18/google-play-store-apps")

Using Colab cache for faster access to the 'google-play-store-apps' dataset.


In [ ]:
df = pd.read_csv("/kaggle/input/google-play-store-apps/googleplaystore.csv")
smart_profiling_report(df)



✅ Smart EDA report saved as Smart_EDA_Report.html


In [ ]:
from google.colab import files
files.download("Smart_EDA_Report.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install plotly jinja2 numpy

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from jinja2 import Template

def smart_profiling_dashboard(df, filename="Smart_EDA_Dashboard.html"):

    # ===== Alerts =====
    alerts = []
    if df.duplicated().sum() > 0:
        alerts.append(f"Duplicate Rows: {df.duplicated().sum()}")

    const_cols = [c for c in df.columns if df[c].nunique() == 1]
    if const_cols:
        alerts.append(f"Constant Columns: {const_cols}")

    miss_cols = df.columns[df.isnull().any()].tolist()
    if miss_cols:
        alerts.append(f"Missing Values Columns: {miss_cols}")

    alerts_html = "<br>".join(alerts) if alerts else "No major warnings"

    # ===== Column Sections =====
    sections = ""
    nav_links = ""

    for col in df.columns:
        nav_links += f'<li><a href="#{col}">{col}</a></li>'

        section_html = f"<h2 id='{col}'>{col}</h2>"

        if df[col].dtype in [np.int64, np.float64]:
            section_html += df[col].describe().to_frame().to_html()

            fig = px.histogram(df, x=col, title=f"{col} Distribution")
            section_html += fig.to_html(full_html=False)

            fig2 = px.box(df, y=col, title=f"{col} Boxplot")
            section_html += fig2.to_html(full_html=False)

        else:
            vc = df[col].value_counts().head(10).reset_index()
            vc.columns = [col, 'Count']
            section_html += vc.to_html()

            fig = px.bar(vc, x=col, y='Count', title=f"{col} Top Values")
            section_html += fig.to_html(full_html=False)

        sections += section_html + "<hr>"

    # ===== Correlation =====
    corr = df.corr(numeric_only=True)
    corr_plot = px.imshow(corr, text_auto=True,
                          title="Correlation Heatmap").to_html(full_html=False)

    # ===== HTML Template =====
    html = f"""
    <html>
    <head>
    <title>Smart EDA Dashboard</title>
    <style>
        body {{font-family: Arial; margin:0;}}
        .sidebar {{
            position: fixed; width: 250px; height: 100%;
            overflow:auto; background:#111; padding:20px;
        }}
        .sidebar a {{color:white; text-decoration:none;}}
        .sidebar li {{margin:8px 0;}}
        .content {{margin-left:270px; padding:20px;}}
        h1 {{color:#333;}}
        h2 {{border-bottom:2px solid #ddd; padding-bottom:5px;}}
        .alerts {{background:#ffe6e6; padding:15px; border-left:6px solid red;}}
    </style>
    </head>
    <body>

    <div class="sidebar">
        <h2 style="color:white;">Variables</h2>
        <ul>{nav_links}</ul>
    </div>

    <div class="content">
        <h1>📊 Smart EDA Profiling Dashboard</h1>

        <div class="alerts">
            <h3>⚠️ Alerts</h3>
            {alerts_html}
        </div>

        <h2>Correlation Heatmap</h2>
        {corr_plot}

        {sections}
    </div>

    </body>
    </html>
    """

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Dashboard saved as {filename}")


In [ ]:
df = pd.read_csv("google_play_store_updated.csv")   # apni file ka naam likho
smart_profiling_dashboard(df)

✅ Dashboard saved as Smart_EDA_Dashboard.html


In [ ]:
from google.colab import files

# Agar tumhara file Smart_EDA_Dashboard.html hai
files.download("Smart_EDA_Dashboard.html")




<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from jinja2 import Template

def smart_profiling_dashboard_v2(df, filename="Smart_EDA_Dashboard_v2.html"):

    # ===== Alerts =====
    alerts = []
    if df.duplicated().sum() > 0:
        alerts.append(f"Duplicate Rows: {df.duplicated().sum()}")

    const_cols = [c for c in df.columns if df[c].nunique() == 1]
    if const_cols:
        alerts.append(f"Constant Columns: {const_cols}")

    miss_cols = df.columns[df.isnull().any()].tolist()
    if miss_cols:
        alerts.append(f"Missing Values Columns: {miss_cols}")

    alerts_html = ""
    if alerts:
        alerts_html = "<br>".join(alerts)
        alerts_html = f"""
        <div class="alerts">
            <h3>⚠️ Alerts</h3>
            {alerts_html}
        </div>
        """

    # ===== Column Sections =====
    sections = ""
    nav_links = ""

    for col in df.columns:
        nav_links += f'<li><a href="#{col}">{col}</a></li>'

        section_html = f"<h2 id='{col}'>{col}</h2>"

        if df[col].dtype in [np.int64, np.float64]:
            section_html += df[col].describe().to_frame().to_html()

            fig = px.histogram(df, x=col, title=f"{col} Distribution",
                               width=600, height=400)
            section_html += fig.to_html(full_html=False)

            fig2 = px.box(df, y=col, title=f"{col} Boxplot",
                          width=600, height=400)
            section_html += fig2.to_html(full_html=False)

        else:
            vc = df[col].value_counts().head(10).reset_index()
            vc.columns = [col, 'Count']
            section_html += vc.to_html()

            fig = px.bar(vc, x=col, y='Count', title=f"{col} Top Values",
                         width=600, height=400)
            section_html += fig.to_html(full_html=False)

        sections += section_html + "<hr>"

    # ===== Correlation =====
    corr = df.corr(numeric_only=True)
    corr_plot = px.imshow(corr, text_auto=True,
                          title="Correlation Heatmap",
                          width=600, height=500).to_html(full_html=False)

    # ===== HTML Template =====
    html = f"""
    <html>
    <head>
    <title>Smart EDA Dashboard v2</title>
    <style>
        body {{font-family: Arial; margin:0;}}
        .sidebar {{
            position: fixed; width: 250px; height: 100%;
            overflow:auto; background:#111; padding:20px;
        }}
        .sidebar a {{color:white; text-decoration:none;}}
        .sidebar li {{margin:8px 0;}}
        .content {{margin-left:270px; padding:20px;}}
        h1 {{color:#333;}}
        h2 {{border-bottom:2px solid #ddd; padding-bottom:5px;}}
        .alerts {{background:#ffe6e6; padding:15px; border-left:6px solid red;}}
        hr {{border:1px solid #ddd;}}
    </style>
    </head>
    <body>

    <div class="sidebar">
        <h2 style="color:white;">Variables</h2>
        <ul>{nav_links}</ul>
    </div>

    <div class="content">
        <h1>📊 Smart EDA Profiling Dashboard v2</h1>

        {alerts_html}

        <h2>Correlation Heatmap</h2>
        {corr_plot}

        {sections}
    </div>

    </body>
    </html>
    """

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Dashboard saved as {filename}")


In [ ]:
df = pd.read_csv("google_play_store_updated.csv")
smart_profiling_dashboard_v2(df)


✅ Dashboard saved as Smart_EDA_Dashboard_v2.html


In [ ]:
from google.colab import files

# Agar tumne function smart_profiling_dashboard_v2 use kiya hai:
files.download("Smart_EDA_Dashboard_v2.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(df["Installs"].dtype)


float64


In [ ]:

import pandas as pd
import plotly.express as px

# Load your dataset
df = pd.read_csv("google_play_store_updated.csv")

# Convert Last Updated to datetime & extract year
df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors='coerce')
df["Year"] = df["Last Updated"].dt.year

# If Installs is already numeric (float64), use it directly
df_anim = df.dropna(subset=["Year", "Installs"])

# Animated Bar Chart (Category-wise installs over years)
fig = px.bar(
    df_anim,
    x="Category",           # Or "Country" if your dataset has it
    y="Installs",           # Numeric metric
    color="Category",
    animation_frame="Year", # Play year by year
    animation_group="Category",
    range_y=[0, df_anim["Installs"].max()],
    title="Play Store Installs by Category Over Years"
)

fig.update_layout(width=900, height=600)
fig.show()

# Save as HTML for sharing or download
fig.write_html("PlayStore_Animated.html")


In [ ]:
from google.colab import files
files.download("PlayStore_Animated.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# New and Latest EDA report

In [ ]:
# pip install pdfkit

In [ ]:
# =============================
# SMART EDA DASHBOARD
# =============================

import pandas as pd
import numpy as np
import plotly.express as px
from jinja2 import Template

def smart_eda_dashboard(df, html_file="Smart_EDA_Clean.html"):

    # ====== Basic Info ======
    shape = df.shape
    dtypes_html = df.dtypes.to_frame("Data Type").to_html()
    describe_html = df.describe(include='all').to_html()

    # ====== Alerts / Warnings ======
    alerts = []

    if df.duplicated().sum() > 0:
        alerts.append(f"Duplicate Rows: {df.duplicated().sum()}")

    const_cols = [c for c in df.columns if df[c].nunique() == 1]
    if const_cols:
        alerts.append(f"Constant Columns: {const_cols}")

    high_card = [c for c in df.columns if df[c].nunique() > 50]
    if high_card:
        alerts.append(f"High Cardinality Columns: {high_card}")

    miss_cols = df.columns[df.isnull().any()].tolist()
    if miss_cols:
        alerts.append(f"Columns with Missing Values: {miss_cols}")

    alerts_html = "<br>".join(alerts) if alerts else "No major warnings detected."
    alerts_html = f"<div style='background:#ffe6e6; padding:10px; border-left:6px solid red;'>{alerts_html}</div>"

    # ====== Missing Values ======
    missing = (df.isnull().sum() / len(df) * 100).reset_index()
    missing.columns = ['Column', 'Missing %']
    fig_missing = px.bar(missing, x='Column', y='Missing %', title="Missing Values %")
    missing_plot = fig_missing.to_html(full_html=False)

    # ====== Correlation ======
    corr = df.corr(numeric_only=True)
    fig_corr = px.imshow(corr, text_auto=True, title="Correlation Heatmap", width=700, height=500)
    corr_plot = fig_corr.to_html(full_html=False)

    # Top correlations
    high_corr = []
    for i in corr.columns:
        for j in corr.columns:
            if i != j and abs(corr.loc[i, j]) > 0.8:
                high_corr.append(f"{i} & {j} : {corr.loc[i,j]:.2f}")

    # ====== Numeric Distributions ======
    histograms = ""
    for col in df.select_dtypes(include=np.number).columns:
        fig = px.histogram(df, x=col, title=f"{col} Distribution", width=600, height=400)
        histograms += fig.to_html(full_html=False)

        fig_box = px.box(df, y=col, title=f"{col} Boxplot", width=600, height=400)
        histograms += fig_box.to_html(full_html=False)

    # ====== Categorical Distributions ======
    bars = ""
    for col in df.select_dtypes(include='object').columns:
        vc = df[col].value_counts().head(10).reset_index()
        vc.columns = [col, 'Count']
        fig = px.bar(vc, x=col, y='Count', title=f"Top {col} Categories", width=600, height=400)
        bars += fig.to_html(full_html=False)

    # ====== HTML Template ======
    html_template = """
    <html>
    <head>
        <title>Smart EDA Dashboard</title>
        <style>
            body {font-family: Arial; margin:0; background:#f9f9f9;}
            .sidebar {position:fixed; width:220px; height:100%; overflow:auto; background:#111; padding:20px;}
            .sidebar a {color:white; text-decoration:none;}
            .sidebar li {margin:8px 0;}
            .content {margin-left:240px; padding:20px;}
            h1 {color:#333;}
            h2 {border-bottom:2px solid #ddd; padding-bottom:5px;}
            hr {border:1px solid #ddd;}
        </style>
    </head>
    <body>
        <div class="sidebar">
            <h2 style="color:white;">Sections</h2>
            <ul>
                <li><a href="#summary">Summary</a></li>
                <li><a href="#alerts">Alerts</a></li>
                <li><a href="#missing">Missing Values</a></li>
                <li><a href="#correlation">Correlation</a></li>
                <li><a href="#numeric">Numeric Distributions</a></li>
                <li><a href="#categorical">Categorical Distributions</a></li>
            </ul>
        </div>

        <div class="content">
            <h1>📊 Smart EDA Dashboard</h1>

            <h2 id="summary">Dataset Summary</h2>
            <p>Shape: {{shape}}</p>
            <h3>Data Types</h3>{{dtypes}}
            <h3>Statistical Summary</h3>{{describe}}

            <h2 id="alerts">⚠️ Alerts & Warnings</h2>
            {{alerts_html}}

            <h2 id="missing">Missing Values</h2>
            {{missing_plot}}

            <h2 id="correlation">Correlation Heatmap</h2>
            {{corr_plot}}
            {% if high_corr %}
            <p><strong>Highly Correlated Pairs:</strong> {{high_corr}}</p>
            {% endif %}

            <h2 id="numeric">Numeric Distributions</h2>
            {{histograms}}

            <h2 id="categorical">Categorical Distributions</h2>
            {{bars}}
        </div>
    </body>
    </html>
    """

    template = Template(html_template)
    html = template.render(
        shape=shape,
        dtypes=dtypes_html,
        describe=describe_html,
        alerts_html=alerts_html,
        missing_plot=missing_plot,
        corr_plot=corr_plot,
        high_corr=", ".join(high_corr),
        histograms=histograms,
        bars=bars
    )

    # ====== Save HTML ======
    with open(html_file, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ HTML Dashboard saved as {html_file}")
    print("ℹ️ You can open this HTML in browser and use 'Print → Save as PDF' to get a PDF.")

# -------------------------------
# ====== USAGE EXAMPLE ======
# -------------------------------
df = pd.read_csv("google_play_store_updated.csv")
smart_eda_dashboard(df, html_file="Smart_EDA_Clean.html")


✅ HTML Dashboard saved as Smart_EDA_Clean.html
ℹ️ You can open this HTML in browser and use 'Print → Save as PDF' to get a PDF.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from jinja2 import Template

def smart_eda_dashboard_clean(df, html_file="Smart_EDA_Clean_Tips.html"):

    # ====== Basic Info ======
    shape = df.shape
    dtypes_html = df.dtypes.to_frame("Data Type").to_html()
    describe_html = df.describe(include='all').to_html()

    # ====== Alerts / Warnings ======
    alerts = []

    if df.duplicated().sum() > 0:
        alerts.append(f"Duplicate Rows: {df.duplicated().sum()}")

    const_cols = [c for c in df.columns if df[c].nunique() == 1]
    if const_cols:
        alerts.append(f"Constant Columns: {const_cols}")

    high_card = [c for c in df.columns if df[c].nunique() > 50]
    if high_card:
        alerts.append(f"High Cardinality Columns: {high_card}")

    miss_cols = df.columns[df.isnull().any()].tolist()
    if miss_cols:
        alerts.append(f"Columns with Missing Values: {miss_cols}")

    alerts_html = "<br>".join(alerts) if alerts else "No major warnings detected."
    alerts_html = f"<div style='background:#ffe6e6; padding:10px; border-left:6px solid red;'>{alerts_html}</div>"

    # ====== Missing Values ======
    missing = (df.isnull().sum() / len(df) * 100).reset_index()
    missing.columns = ['Column', 'Missing %']
    fig_missing = px.bar(
        missing, x='Column', y='Missing %', title="Missing Values %",
        text_auto=True, width=700, height=400
    )
    missing_plot = fig_missing.to_html(full_html=False)

    # ====== Correlation ======
    corr = df.corr(numeric_only=True)
    fig_corr = px.imshow(corr, text_auto=True, title="Correlation Heatmap", width=700, height=500)
    corr_plot = fig_corr.to_html(full_html=False)

    # Top correlations
    high_corr = []
    for i in corr.columns:
        for j in corr.columns:
            if i != j and abs(corr.loc[i, j]) > 0.8:
                high_corr.append(f"{i} & {j} : {corr.loc[i,j]:.2f}")

    # ====== Numeric Distributions ======
    histograms = ""
    for col in df.select_dtypes(include=np.number).columns:
        # Numeric histogram with hover tips for key stats
        fig = px.histogram(
            df, x=col, title=f"{col} Distribution", width=600, height=400,
            marginal="box",  # shows boxplot below histogram automatically
            hover_data={col: True}  # shows value on hover
        )
        histograms += fig.to_html(full_html=False)

    # ====== Categorical Distributions ======
    bars = ""
    for col in df.select_dtypes(include='object').columns:
        vc = df[col].value_counts().head(10).reset_index()
        vc.columns = [col, 'Count']
        fig = px.bar(
            vc, x=col, y='Count', title=f"Top {col} Categories",
            text_auto=True, width=600, height=400
        )
        bars += fig.to_html(full_html=False)

    # ====== HTML Template ======
    html_template = """
    <html>
    <head>
        <title>Smart EDA Dashboard - Tips</title>
        <style>
            body {font-family: Arial; margin:0; background:#f9f9f9;}
            .sidebar {position:fixed; width:220px; height:100%; overflow:auto; background:#111; padding:20px;}
            .sidebar a {color:white; text-decoration:none;}
            .sidebar li {margin:8px 0;}
            .content {margin-left:240px; padding:20px;}
            h1 {color:#333;}
            h2 {border-bottom:2px solid #ddd; padding-bottom:5px;}
            hr {border:1px solid #ddd;}
        </style>
    </head>
    <body>
        <div class="sidebar">
            <h2 style="color:white;">Sections</h2>
            <ul>
                <li><a href="#summary">Summary</a></li>
                <li><a href="#alerts">Alerts</a></li>
                <li><a href="#missing">Missing Values</a></li>
                <li><a href="#correlation">Correlation</a></li>
                <li><a href="#numeric">Numeric Distributions</a></li>
                <li><a href="#categorical">Categorical Distributions</a></li>
            </ul>
        </div>

        <div class="content">
            <h1>📊 Smart EDA Dashboard - Tips</h1>

            <h2 id="summary">Dataset Summary</h2>
            <p>Shape: {{shape}}</p>
            <h3>Data Types</h3>{{dtypes}}
            <h3>Statistical Summary</h3>{{describe}}

            <h2 id="alerts">⚠️ Alerts & Warnings</h2>
            {{alerts_html}}

            <h2 id="missing">Missing Values</h2>
            {{missing_plot}}

            <h2 id="correlation">Correlation Heatmap</h2>
            {{corr_plot}}
            {% if high_corr %}
            <p><strong>Highly Correlated Pairs:</strong> {{high_corr}}</p>
            {% endif %}

            <h2 id="numeric">Numeric Distributions</h2>
            {{histograms}}

            <h2 id="categorical">Categorical Distributions</h2>
            {{bars}}
        </div>
    </body>
    </html>
    """

    template = Template(html_template)
    html = template.render(
        shape=shape,
        dtypes=dtypes_html,
        describe=describe_html,
        alerts_html=alerts_html,
        missing_plot=missing_plot,
        corr_plot=corr_plot,
        high_corr=", ".join(high_corr),
        histograms=histograms,
        bars=bars
    )

    # ====== Save HTML ======
    with open(html_file, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ HTML Dashboard saved as {html_file}")
    print("ℹ️ Open this HTML in browser and use 'Print → Save as PDF' to export PDF.")

# -------------------------------
# ====== USAGE EXAMPLE ======
# -------------------------------
df = pd.read_csv("google_play_store_updated.csv")
smart_eda_dashboard_clean(df)


✅ HTML Dashboard saved as Smart_EDA_Clean_Tips.html
ℹ️ Open this HTML in browser and use 'Print → Save as PDF' to export PDF.


In [ ]:
# =========================================
# SMART EDA DASHBOARD - FULL FEATURED VERSION
# =========================================

import pandas as pd
import numpy as np
import plotly.express as px
from jinja2 import Template

def smart_eda_dashboard_pro(df, html_file="Smart_EDA_Pro.html"):

    # ====== Dataset Summary Cards ======
    total_rows = df.shape[0]
    total_cols = df.shape[1]
    total_missing = df.isnull().sum().sum()
    total_duplicates = df.duplicated().sum()
    const_cols = [c for c in df.columns if df[c].nunique() == 1]
    high_card = [c for c in df.columns if df[c].nunique() > 50]

    cards_html = f"""
    <div style='display:flex; gap:20px; flex-wrap:wrap;'>
        <div style='background:#4CAF50;color:white;padding:15px;border-radius:8px;flex:1;'>
            <h3>Total Rows</h3>
            <p>{total_rows}</p>
        </div>
        <div style='background:#2196F3;color:white;padding:15px;border-radius:8px;flex:1;'>
            <h3>Total Columns</h3>
            <p>{total_cols}</p>
        </div>
        <div style='background:#FF9800;color:white;padding:15px;border-radius:8px;flex:1;'>
            <h3>Total Missing</h3>
            <p>{total_missing}</p>
        </div>
        <div style='background:#f44336;color:white;padding:15px;border-radius:8px;flex:1;'>
            <h3>Duplicate Rows</h3>
            <p>{total_duplicates}</p>
        </div>
    </div>
    <br>
    """

    # ====== Alerts / Warnings ======
    alerts = []
    if const_cols:
        alerts.append(f"Constant Columns: {const_cols}")
    if high_card:
        alerts.append(f"High Cardinality Columns: {high_card}")
    if total_duplicates > 0:
        alerts.append(f"Duplicate Rows: {total_duplicates}")
    if total_missing > 0:
        alerts.append(f"Columns with Missing Values: {df.columns[df.isnull().any()].tolist()}")

    alerts_html = "<br>".join(alerts) if alerts else "No major warnings detected."
    alerts_html = f"<div style='background:#ffe6e6; padding:10px; border-left:6px solid red;'>{alerts_html}</div>"

    # ====== Missing Values Chart ======
    missing = (df.isnull().sum() / len(df) * 100).reset_index()
    missing.columns = ['Column', 'Missing %']
    fig_missing = px.bar(missing, x='Column', y='Missing %', title="Missing Values (%)", text_auto=True, width=700, height=400)
    missing_plot = fig_missing.to_html(full_html=False)

    # ====== Correlation Heatmap ======
    corr = df.corr(numeric_only=True)
    fig_corr = px.imshow(corr, text_auto=True, title="Correlation Heatmap", width=700, height=500)
    corr_plot = fig_corr.to_html(full_html=False)

    high_corr_pairs = []
    for i in corr.columns:
        for j in corr.columns:
            if i != j and abs(corr.loc[i,j]) > 0.8:
                high_corr_pairs.append(f"{i} & {j}: {corr.loc[i,j]:.2f}")
    high_corr_text = ", ".join(high_corr_pairs) if high_corr_pairs else "None"

    # ====== Numeric Distributions & Tables ======
    numeric_sections = ""
    for col in df.select_dtypes(include=np.number).columns:
        # Histogram + boxplot
        fig = px.histogram(df, x=col, title=f"{col} Distribution", marginal="box", width=600, height=400, hover_data={col: True})
        numeric_sections += fig.to_html(full_html=False)

        # Top 5 / Bottom 5 table
        top5 = df[col].nlargest(5).to_frame().reset_index().rename(columns={'index':'Index', col: col})
        bottom5 = df[col].nsmallest(5).to_frame().reset_index().rename(columns={'index':'Index', col: col})
        table_html = f"<h4>{col} Top 5 Values</h4>{top5.to_html(index=False)}<h4>{col} Bottom 5 Values</h4>{bottom5.to_html(index=False)}<hr>"
        numeric_sections += table_html

    # ====== Categorical Distributions & Tables ======
    categorical_sections = ""
    for col in df.select_dtypes(include='object').columns:
        vc = df[col].value_counts().head(10).reset_index()
        vc.columns = [col, 'Count']
        fig = px.bar(vc, x=col, y='Count', text_auto=True, title=f"Top {col} Categories", width=600, height=400)
        categorical_sections += fig.to_html(full_html=False)

        # Top 10 table
        table_html = f"<h4>{col} Top 10 Categories</h4>{vc.to_html(index=False)}<hr>"
        categorical_sections += table_html

    # ====== HTML Template ======
    html_template = """
    <html>
    <head>
        <title>Smart EDA Dashboard Pro</title>
        <style>
            body {font-family: Arial; margin:0; background:#f9f9f9;}
            .sidebar {position:fixed; width:220px; height:100%; overflow:auto; background:#111; padding:20px;}
            .sidebar a {color:white; text-decoration:none;}
            .sidebar li {margin:8px 0;}
            .content {margin-left:240px; padding:20px;}
            h1 {color:#333;}
            h2 {border-bottom:2px solid #ddd; padding-bottom:5px;}
            hr {border:1px solid #ddd;}
        </style>
    </head>
    <body>
        <div class="sidebar">
            <h2 style="color:white;">Sections</h2>
            <ul>
                <li><a href="#cards">Summary Cards</a></li>
                <li><a href="#alerts">Alerts</a></li>
                <li><a href="#missing">Missing Values</a></li>
                <li><a href="#correlation">Correlation</a></li>
                <li><a href="#numeric">Numeric Columns</a></li>
                <li><a href="#categorical">Categorical Columns</a></li>
            </ul>
        </div>

        <div class="content">
            <h1>📊 Smart EDA Dashboard Pro</h1>

            <h2 id="cards">Dataset Summary Cards</h2>
            {{cards_html}}

            <h2 id="alerts">⚠️ Alerts & Warnings</h2>
            {{alerts_html}}

            <h2 id="missing">Missing Values</h2>
            {{missing_plot}}

            <h2 id="correlation">Correlation Heatmap</h2>
            {{corr_plot}}
            <p><strong>Highly Correlated Pairs:</strong> {{high_corr_text}}</p>

            <h2 id="numeric">Numeric Columns</h2>
            {{numeric_sections}}

            <h2 id="categorical">Categorical Columns</h2>
            {{categorical_sections}}
        </div>
    </body>
    </html>
    """

    template = Template(html_template)
    html = template.render(
        cards_html=cards_html,
        alerts_html=alerts_html,
        missing_plot=missing_plot,
        corr_plot=corr_plot,
        high_corr_text=high_corr_text,
        numeric_sections=numeric_sections,
        categorical_sections=categorical_sections
    )

    # ====== Save HTML ======
    with open(html_file, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Full-featured HTML Dashboard saved as {html_file}")
    print("ℹ️ Open this HTML in browser → Print → Save as PDF for PDF report")

# -------------------------------
# ====== USAGE EXAMPLE ======
# -------------------------------
df = pd.read_csv("google_play_store_updated.csv")
smart_eda_dashboard_pro(df)


✅ Full-featured HTML Dashboard saved as Smart_EDA_Pro.html
ℹ️ Open this HTML in browser → Print → Save as PDF for PDF report


# new EDA

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from jinja2 import Template

def smart_eda_dashboard_next(df, html_file="Smart_EDA_Next.html"):

    # ===== KPI CARDS =====
    total_rows = df.shape[0]
    total_cols = df.shape[1]
    total_missing = df.isnull().sum().sum()
    total_duplicates = df.duplicated().sum()
    const_cols = [c for c in df.columns if df[c].nunique() == 1]
    high_card = [c for c in df.columns if df[c].nunique() > 50]

    cards_html = f"""
    <div style='display:flex; gap:20px; flex-wrap:wrap;'>
        <div style='background:#4CAF50;color:white;padding:15px;border-radius:10px;flex:1; text-align:center;'>
            <h3>🗂 Total Rows</h3>
            <p>{total_rows}</p>
        </div>
        <div style='background:#2196F3;color:white;padding:15px;border-radius:10px;flex:1; text-align:center;'>
            <h3>📊 Total Columns</h3>
            <p>{total_cols}</p>
        </div>
        <div style='background:#FF9800;color:white;padding:15px;border-radius:10px;flex:1; text-align:center;'>
            <h3>⚠ Missing Values</h3>
            <p>{total_missing}</p>
        </div>
        <div style='background:#f44336;color:white;padding:15px;border-radius:10px;flex:1; text-align:center;'>
            <h3>🧩 Duplicates</h3>
            <p>{total_duplicates}</p>
        </div>
    </div>
    <br>
    """

    # ===== ALERTS =====
    alerts = []
    if const_cols:
        alerts.append(f"Constant Columns: {const_cols}")
    if high_card:
        alerts.append(f"High Cardinality Columns: {high_card}")
    if total_duplicates > 0:
        alerts.append(f"Duplicate Rows: {total_duplicates}")
    if total_missing > 0:
        alerts.append(f"Columns with Missing Values: {df.columns[df.isnull().any()].tolist()}")

    alerts_html = "<br>".join(alerts) if alerts else "No major warnings detected."
    alerts_html = f"<div style='background:#ffe6e6; padding:12px; border-left:6px solid red;'>{alerts_html}</div>"

    # ===== MISSING VALUES =====
    missing = (df.isnull().sum() / len(df) * 100).reset_index()
    missing.columns = ['Column', 'MissingPercent']
    fig_missing = px.bar(
        missing, x='Column', y='MissingPercent', title="Missing Values (%)",
        text_auto=True, width=700, height=400
    )
    missing_plot = fig_missing.to_html(full_html=False)

    # ===== CORRELATION =====
    corr = df.corr(numeric_only=True)
    fig_corr = px.imshow(
        corr, text_auto=True, title="Correlation Heatmap", width=700, height=500, color_continuous_scale="Viridis"
    )
    corr_plot = fig_corr.to_html(full_html=False)

    high_corr_pairs = []
    for i in corr.columns:
        for j in corr.columns:
            if i != j and abs(corr.loc[i,j]) > 0.8:
                high_corr_pairs.append(f"{i} & {j} = {corr.loc[i,j]:.2f}")
    high_corr_text = ", ".join(high_corr_pairs) if high_corr_pairs else "None"

    # ===== NUMERIC SECTIONS =====
    numeric_sections = ""
    for col in df.select_dtypes(include=np.number).columns:
        numeric_sections += f"""
        <details>
            <summary style='font-size:18px; font-weight:bold; cursor:pointer;'>{col} (Numeric)</summary>
        """
        fig = px.histogram(df, x=col, marginal="box", width=650, height=400, hover_data={col: True})
        numeric_sections += fig.to_html(full_html=False)

        top5 = df[col].nlargest(5).to_frame().reset_index().rename(columns={'index':'Index', col: col})
        bottom5 = df[col].nsmallest(5).to_frame().reset_index().rename(columns={'index':'Index', col: col})
        numeric_sections += "<h4>Top 5 Values</h4>" + top5.to_html(index=False)
        numeric_sections += "<h4>Bottom 5 Values</h4>" + bottom5.to_html(index=False) + "<hr></details>"

    # ===== CATEGORICAL SECTIONS =====
    categorical_sections = ""
    for col in df.select_dtypes(include='object').columns:
        categorical_sections += f"""
        <details>
            <summary style='font-size:18px; font-weight:bold; cursor:pointer;'>{col} (Categorical)</summary>
        """
        vc = df[col].value_counts().head(10).reset_index()
        vc.columns = [col, 'Count']
        fig = px.bar(vc, x=col, y='Count', text_auto=True, width=650, height=400)
        categorical_sections += fig.to_html(full_html=False)
        categorical_sections += "<h4>Top 10 Categories Table</h4>" + vc.to_html(index=False) + "<hr></details>"

    # ===== HTML TEMPLATE (SAFE) =====
    html_template = """
    <html>
    <head>
        <title>Smart EDA Dashboard Next-Level</title>
        <style>
            body {font-family: Arial; margin:0; background:#f9f9f9;}
            .sidebar {position:fixed; width:240px; height:100%; overflow:auto; background:#111; padding:20px;}
            .sidebar a {color:white; text-decoration:none;}
            .sidebar li {margin:10px 0;}
            .content {margin-left:260px; padding:20px;}
            h1 {color:#333;}
            h2 {border-bottom:2px solid #ddd; padding-bottom:5px;}
            details summary {background:#eee; padding:8px; border-radius:5px;}
            details summary:hover {background:#ddd;}
            hr {border:1px solid #ddd;}
        </style>
    </head>
    <body>
        <div class="sidebar">
            <h2 style="color:white;">Sections</h2>
            <ul>
                <li><a href="#cards">Summary Cards</a></li>
                <li><a href="#alerts">Alerts</a></li>
                <li><a href="#missing">Missing Values</a></li>
                <li><a href="#correlation">Correlation</a></li>
                <li><a href="#numeric">Numeric Columns</a></li>
                <li><a href="#categorical">Categorical Columns</a></li>
            </ul>
        </div>

        <div class="content">
            <h1>📊 Smart EDA Dashboard Next-Level</h1>

            <h2 id="cards">Dataset Summary Cards</h2>
            {{ cards_html }}

            <h2 id="alerts">⚠️ Alerts & Warnings</h2>
            {{ alerts_html }}

            <h2 id="missing">Missing Values</h2>
            {{ missing_plot }}

            <h2 id="correlation">Correlation Heatmap</h2>
            {{ corr_plot }}
            <p><strong>Highly Correlated Pairs</strong> {{ high_corr_text }}</p>

            <h2 id="numeric">Numeric Columns</h2>
            {{ numeric_sections }}

            <h2 id="categorical">Categorical Columns</h2>
            {{ categorical_sections }}
        </div>
    </body>
    </html>
    """

    template = Template(html_template)
    html = template.render(
        cards_html=cards_html,
        alerts_html=alerts_html,
        missing_plot=missing_plot,
        corr_plot=corr_plot,
        high_corr_text=high_corr_text,
        numeric_sections=numeric_sections,
        categorical_sections=categorical_sections
    )

    with open(html_file, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Next-Level HTML Dashboard saved as {html_file}")
    print("ℹ️ Open this HTML in browser → Print → Save as PDF for PDF report")

# -------------------------------
# USAGE
# -------------------------------
df = pd.read_csv("google_play_store_updated.csv")
smart_eda_dashboard_next(df)


✅ Next-Level HTML Dashboard saved as Smart_EDA_Next.html
ℹ️ Open this HTML in browser → Print → Save as PDF for PDF report


# Eda_next_v2

In [ ]:
# !pip install streamlit pandas matplotlib seaborn
# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
# !dpkg -i cloudflared-linux-amd64.deb


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

st.set_page_config(layout="wide")
st.title("EDA_Next v2 Dashboard")

file = st.file_uploader("google_play_store_updated.csv", type=["csv"])

if file:
    df = pd.read_csv(file)

    st.write("### Data Preview")
    st.dataframe(df.head())

    st.write("### Correlation Heatmap")
    corr = df.corr(numeric_only=True)
    if not corr.empty:
        fig, ax = plt.subplots(figsize=(8,5))
        sns.heatmap(corr, ax=ax)
        st.pyplot(fig)


Overwriting app.py


In [ ]:
import subprocess, time
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(5)

In [ ]:
# !cloudflared tunnel --url http://localhost:8501

In [ ]:
# !pip install pandas plotly ipywidgets


In [ ]:
# Install required libraries
# !pip install pandas plotly ipywidgets

import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# Sample dataset
df = pd.read_csv("google_play_store_updated.csv")

st_title = "📊 EDA_Next v4 — Demo Dashboard ( Dataset)"
print(st_title)
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Sidebar filters
filters = {}
for col in df.columns:
    if df[col].dtype=='object':
        vals = df[col].unique().tolist()
        sel = widgets.SelectMultiple(
            options=vals,
            value=vals[:min(5,len(vals))],
            description=col
        )
        display(sel)
        filters[col] = sel
    else:
        slider = widgets.FloatRangeSlider(
            value=[df[col].min(), df[col].max()],
            min=df[col].min(),
            max=df[col].max(),
            step=(df[col].max()-df[col].min())/100,
            description=col
        )
        display(slider)
        filters[col] = slider

# Function to apply filters
def apply_filters(btn):
    df_filtered = df.copy()
    for col, w in filters.items():
        if df[col].dtype=='object':
            df_filtered = df_filtered[df_filtered[col].isin(w.value)]
        else:
            df_filtered = df_filtered[(df_filtered[col]>=w.value[0]) & (df_filtered[col]<=w.value[1])]
    display_dashboard(df_filtered)

# Function to display dashboard
def display_dashboard(df_filtered):
    print("\n✅ Filtered Dataset Preview:")
    display(df_filtered.head())

    # Top correlations
    numeric_cols = df_filtered.select_dtypes(include=['int64','float64']).columns
    if len(numeric_cols)>1:
        corr = df_filtered[numeric_cols].corr()
        print("\n🔥 Top Correlations:")
        display(corr)
        fig = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale="RdBu_r")
        fig.show()

    # Outliers
    print("\n📦 Top Outliers:")
    for col in numeric_cols:
        top_out = df_filtered[col].sort_values(ascending=False).head(5).values
        print(f"{col}:", top_out)

    # Interactive plots
    for col in numeric_cols:
        fig = px.histogram(df_filtered, x=col, title=f"Distribution of {col}")
        fig.show()

# Button to apply filters
apply_btn = widgets.Button(description="Apply Filters & Show Dashboard")
apply_btn.on_click(apply_filters)
display(apply_btn)


📊 EDA_Next v4 — Demo Dashboard ( Dataset)
Rows: 10346, Columns: 15


SelectMultiple(description='App', index=(0, 1, 2, 3, 4), options=('Photo Editor & Candy Camera & Grid & ScrapB…

SelectMultiple(description='Category', index=(0, 1, 2, 3, 4), options=('ART_AND_DESIGN', 'AUTO_AND_VEHICLES', …

FloatRangeSlider(value=(0.0, 5.0), description='Rating', max=5.0, step=0.05)

FloatRangeSlider(value=(0.0, 78158306.0), description='Reviews', max=78158306.0, step=781583.06)

...

,App,Category,Rating,Reviews,Size_in_bytes,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Installs_category,Size_in_Mb



🔥 Top Correlations:


,Rating,Reviews,Size_in_bytes,Installs,Price,Size_in_Mb
Rating,NaN,NaN,NaN,NaN,NaN,NaN
Reviews,NaN,NaN,NaN,NaN,NaN,NaN
Size_in_bytes,NaN,NaN,NaN,NaN,NaN,NaN
Installs,NaN,NaN,NaN,NaN,NaN,NaN
Price,NaN,NaN,NaN,NaN,NaN,NaN
Size_in_Mb,NaN,NaN,NaN,NaN,NaN,NaN



📦 Top Outliers:
Rating: []
Reviews: []
Size_in_bytes: []
Installs: []
Price: []
Size_in_Mb: []


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
# Install libraries
# !pip install pandas plotly

import pandas as pd
import plotly.express as px
from IPython.display import display

# Load sample dataset
df = pd.read_csv("google_play_store_updated.csv")
print("📊 EDA Demo Dashboard — Playstore Dataset")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

# ===== Missing Values =====
missing = df.isnull().sum()
print("🟡 Missing Values per Column:")
display(missing)

# ===== Top Correlations =====
numeric_cols = df.select_dtypes(include=['float64','int64']).columns
if len(numeric_cols)>1:
    corr = df[numeric_cols].corr()
    print("\n🔥 Correlation Matrix:")
    display(corr)
    fig_corr = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", aspect="auto", title="Correlation Heatmap")
    fig_corr.show()

# ===== Outlier Detection =====
print("\n📦 Top 3 Outliers per Numeric Column:")
for col in numeric_cols:
    top_out = df[col].sort_values(ascending=False).head(3).values
    print(f"{col}:", top_out)

# ===== Plots =====
print("\n📊 Histograms of Numeric Columns:")
for col in numeric_cols:
    fig = px.histogram(df, x=col, title=f"Distribution of {col}", nbins=20)
    fig.show()

# ===== Scatter Plot Example =====
print("\n🔹 Example Scatter Plot (Sepal Width vs Sepal Length):")
fig_scatter = px.scatter(df, x="sepal_length", y="sepal_width", color="species", size="petal_length", title="Sepal Width vs Sepal Length")
fig_scatter.show()


📊 EDA Demo Dashboard — Playstore Dataset
Rows: 10346, Columns: 15

🟡 Missing Values per Column:


,0
App,0
Category,0
Rating,0
Reviews,0
Size_in_bytes,0
Installs,0
Type,0
Price,0
Content Rating,0
Genres,0



🔥 Correlation Matrix:


,Rating,Reviews,Size_in_bytes,Installs,Price,Size_in_Mb
Rating,1.000000,0.079550,0.102537,0.083886,-0.016277,0.102537
Reviews,0.079550,1.000000,0.070434,0.634987,-0.009424,0.070434
Size_in_bytes,0.102537,0.070434,1.000000,0.000170,-0.015086,1.000000
Installs,0.083886,0.634987,0.000170,1.000000,-0.011155,0.000170
Price,-0.016277,-0.009424,-0.015086,-0.011155,1.000000,-0.015086
Size_in_Mb,0.102537,0.070434,1.000000,0.000170,-0.015086,1.000000



📦 Top 3 Outliers per Numeric Column:
Rating: [5. 5. 5.]
Reviews: [78158306. 78128208. 69119316.]
Size_in_bytes: [1.048576e+08 1.048576e+08 1.048576e+08]
Installs: [1.e+09 1.e+09 1.e+09]
Price: [400.   399.99 399.99]
Size_in_Mb: [100. 100. 100.]

📊 Histograms of Numeric Columns:



🔹 Example Scatter Plot (Sepal Width vs Sepal Length):


ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['App', 'Category', 'Rating', 'Reviews', 'Size_in_bytes', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver', 'Installs_category', 'Size_in_Mb'] but received: sepal_length

In [ ]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns
print(numeric_cols)


Index(['Rating', 'Reviews', 'Size_in_bytes', 'Installs', 'Price',
       'Size_in_Mb'],
      dtype='object')


In [ ]:
# Example scatter using numeric columns from your dataset
if len(numeric_cols) >= 2:
    fig_scatter = px.scatter(
        df,
        x=numeric_cols[0],
        y=numeric_cols[1],
        color=numeric_cols[1],  # optional coloring
        title=f"{numeric_cols[1]} vs {numeric_cols[0]}"
    )
    fig_scatter.show()
else:
    print("Dataset me 2 numeric columns nahi hain scatter plot ke liye.")
